In [1]:
import pandas as pd
import numpy as np

# ============================================================
# FILE
# ============================================================

DATA_FILE = "아모레퍼시픽_Copilot_전사더미데이터_raw_v4(직군세분화).xlsx"

# 최종 Individual Resistance 12개
INDIVIDUAL_FACTORS = [
    "Need Relevance Gap",
    "Value Awareness Gap",
    "Value Experience Gap",
    "Comparative Value Gap",
    "Feature Knowledge Gap",
    "Prompting Gap",
    "Task Application Gap",
    "AI Output Trust Gap",
    "Security Concern",
    "Responsibility Concern",
    "Routine Inertia",
    "Switching Cost",
]

# 기본 결합 가중치
PULSE_WEIGHT = 0.70
BEHAVIOR_WEIGHT = 0.30

In [2]:
def load_diagnosis_data(file_path=DATA_FILE):
    """
    개인 Resistance 진단에 필요한 데이터 로드
    """

    data = {
        "employees": pd.read_excel(
            file_path,
            sheet_name="직원마스터"
        ),

        "adoption": pd.read_excel(
            file_path,
            sheet_name="Adoption_Log"
        ),

        "pulse": pd.read_excel(
            file_path,
            sheet_name="Pulse_Check"
        ),

        "question_map": pd.read_excel(
            file_path,
            sheet_name="Pulse_문항맵"
        ),

        "factor_def": pd.read_excel(
            file_path,
            sheet_name="저항요인_진단기준"
        ),

        "module_map": pd.read_excel(
            file_path,
            sheet_name="처방_모듈맵"
        ),
    }

    return data


data = load_diagnosis_data()

print("데이터 로드 완료")
print("직원:", len(data["employees"]))
print("Pulse:", len(data["pulse"]))
print("Adoption:", len(data["adoption"]))

데이터 로드 완료
직원: 4748
Pulse: 4748
Adoption: 4748


In [3]:
def likert_to_resistance(value, direction):
    """
    Likert 1~5를 Resistance 0~100으로 변환.

    높은 응답이 높은 저항이면:
    1 -> 0
    5 -> 100

    높은 응답이 낮은 저항이면:
    1 -> 100
    5 -> 0
    """

    if pd.isna(value):
        return np.nan

    value = float(value)

    if str(direction).startswith("정방향"):
        return (value - 1) / 4 * 100

    else:
        return (5 - value) / 4 * 100

In [4]:
def calculate_individual_pulse_scores(
    employee_id,
    data
):
    """
    Pulse_문항맵을 이용하여
    해당 직원의 12개 Individual Resistance Pulse Score 계산.
    """

    pulse_df = data["pulse"]
    question_map = data["question_map"]

    # -----------------------------
    # 직원 Pulse 응답 찾기
    # -----------------------------

    employee_pulse = pulse_df[
        (pulse_df["직원ID"] == employee_id) &
        (pulse_df["응답여부"] == "Y")
    ]

    if employee_pulse.empty:
        raise ValueError(
            f"{employee_id}: Pulse 응답 데이터가 없습니다."
        )

    # 가장 최근 행 1개 사용
    employee_pulse = employee_pulse.iloc[-1]

    # -----------------------------
    # Individual Factor만 필터링
    # -----------------------------

    individual_map = question_map[
        question_map["세부Factor"].isin(
            INDIVIDUAL_FACTORS
        )
    ].copy()

    results = []

    # -----------------------------
    # Factor별 계산
    # -----------------------------

    for factor in INDIVIDUAL_FACTORS:

        factor_questions = individual_map[
            individual_map["세부Factor"] == factor
        ]

        weighted_scores = []
        weights = []

        for _, q in factor_questions.iterrows():

            q_number = int(q["문항번호"])
            q_column = f"Q{q_number}"

            if q_column not in employee_pulse.index:
                continue

            raw_value = employee_pulse[q_column]

            score = likert_to_resistance(
                raw_value,
                q["방향"]
            )

            weight = (
                float(q["가중치"])
                if not pd.isna(q["가중치"])
                else 1.0
            )

            if not pd.isna(score):

                weighted_scores.append(
                    score * weight
                )

                weights.append(weight)

        if len(weights) == 0:
            pulse_score = np.nan

        else:
            pulse_score = (
                sum(weighted_scores)
                / sum(weights)
            )

        gap = None

        if not factor_questions.empty:
            gap = factor_questions.iloc[0][
                "상위Gap"
            ]

        results.append({
            "employee_id": employee_id,
            "gap": gap,
            "factor": factor,
            "pulse_score": pulse_score
        })

    return pd.DataFrame(results)

In [5]:
def positive_behavior_to_resistance(
    value,
    goal
):
    """
    좋은 행동이 많을수록 Resistance 감소.

    value = 0
        -> Resistance 100

    value >= goal
        -> Resistance 0

    goal은 PoC 가정값.
    """

    if pd.isna(value):
        return np.nan

    score = 100 * (
        1 - float(value) / goal
    )

    return max(
        0,
        min(100, score)
    )

In [6]:
def calculate_individual_behavior_scores(
    employee_id,
    data
):
    """
    Adoption Log에서 행동으로 관측 가능한
    Individual Resistance만 계산.
    """

    adoption_df = data["adoption"]

    row = adoption_df[
        adoption_df["직원ID"] == employee_id
    ]

    if row.empty:
        raise ValueError(
            f"{employee_id}: Adoption Log가 없습니다."
        )

    row = row.iloc[-1]

    # ========================================================
    # Feature diversity
    # ========================================================

    feature_columns = [
        "기능_초안",
        "기능_회의록",
        "기능_분석",
        "기능_이미지"
    ]

    feature_diversity = sum(
        pd.to_numeric(
            pd.Series([row[c]]),
            errors="coerce"
        ).iloc[0] > 0
        for c in feature_columns
    )

    # ========================================================
    # App diversity
    # ========================================================

    app_columns = [
        "앱_CopilotChat",
        "앱_Word",
        "앱_PowerPoint",
        "앱_Excel",
        "앱_Outlook"
    ]

    app_diversity = sum(
        pd.to_numeric(
            pd.Series([row[c]]),
            errors="coerce"
        ).iloc[0] > 0
        for c in app_columns
    )

    # ========================================================
    # Resistance Proxy
    # ========================================================

    feature_knowledge_score = np.nanmean([

        positive_behavior_to_resistance(
            feature_diversity,
            goal=4
        ),

        positive_behavior_to_resistance(
            app_diversity,
            goal=5
        )
    ])

    task_application_score = (
        positive_behavior_to_resistance(
            row["AI활용업무비율(%)"],
            goal=60
        )
    )

    routine_inertia_score = np.nanmean([

        positive_behavior_to_resistance(
            row["활성일수"],
            goal=16
        ),

        positive_behavior_to_resistance(
            row["지속사용주(8주)"],
            goal=8
        )
    ])

    results = {
        factor: np.nan
        for factor in INDIVIDUAL_FACTORS
    }

    results[
        "Feature Knowledge Gap"
    ] = feature_knowledge_score

    results[
        "Task Application Gap"
    ] = task_application_score

    results[
        "Routine Inertia"
    ] = routine_inertia_score

    return results

In [7]:
def combine_individual_scores(
    pulse_scores,
    behavior_scores,
    pulse_weight=0.70,
    behavior_weight=0.30
):
    """
    Pulse + Behavior 결합.

    행동 Proxy가 없는 Factor:
        Pulse 100%

    행동 Proxy가 있는 Factor:
        Pulse 70% + Behavior 30%
    """

    result = pulse_scores.copy()

    final_scores = []
    behavior_values = []

    for _, row in result.iterrows():

        factor = row["factor"]
        pulse_score = row["pulse_score"]

        behavior_score = (
            behavior_scores.get(
                factor,
                np.nan
            )
        )

        behavior_values.append(
            behavior_score
        )

        if pd.isna(pulse_score):

            final_score = behavior_score

        elif pd.isna(behavior_score):

            # 행동 Proxy 없음
            final_score = pulse_score

        else:

            final_score = (
                pulse_score * pulse_weight
                +
                behavior_score
                * behavior_weight
            )

        final_scores.append(
            final_score
        )

    result[
        "behavior_score"
    ] = behavior_values

    result[
        "resistance_score"
    ] = final_scores

    return result.sort_values(
        "resistance_score",
        ascending=False
    ).reset_index(drop=True)

In [8]:
def diagnose_individual(
    employee_id,
    data
):
    """
    직원 1명의 Individual Resistance 자동 진단.
    """

    pulse_scores = (
        calculate_individual_pulse_scores(
            employee_id,
            data
        )
    )

    behavior_scores = (
        calculate_individual_behavior_scores(
            employee_id,
            data
        )
    )

    scores = combine_individual_scores(
        pulse_scores,
        behavior_scores
    )

    valid_scores = scores.dropna(
        subset=["resistance_score"]
    )

    if valid_scores.empty:
        raise ValueError(
            "계산 가능한 Resistance가 없습니다."
        )

    primary = valid_scores.iloc[0]

    return {
        "employee_id": employee_id,

        "primary_gap":
            primary["gap"],

        "primary_resistance":
            primary["factor"],

        "resistance_score":
            round(
                float(
                    primary[
                        "resistance_score"
                    ]
                ),
                1
            ),

        "all_scores":
            scores
    }

In [9]:
def match_module(
    primary_resistance,
    data
):
    """
    처방_모듈맵에서
    Primary Resistance에 해당하는 Module 자동 매칭.
    """

    module_map = data["module_map"]

    match = module_map[
        module_map["세부Factor"]
        == primary_resistance
    ]

    if match.empty:
        raise ValueError(
            f"{primary_resistance}에 대응하는 "
            "Module이 없습니다."
        )

    row = match.iloc[0]

    return {
        "factor":
            row["세부Factor"],

        "gap":
            row["상위Gap"],

        "module_name":
            row["처방Module"],

        "core_approach":
            row["핵심접근법"],

        "target":
            row["적용대상"]
    }

In [13]:
employee_id = "AP0002"

diagnosis = diagnose_individual(
    employee_id,
    data
)

module = match_module(
    diagnosis["primary_resistance"],
    data
)

print("====================================")
print("INDIVIDUAL AI RESISTANCE DIAGNOSIS")
print("====================================")

print("Employee:", diagnosis["employee_id"])
print("Primary Gap:", diagnosis["primary_gap"])
print("Primary Resistance:", diagnosis["primary_resistance"])
print("Resistance Score:", diagnosis["resistance_score"])
print("Assigned Module:", module["module_name"])
print("Core Approach:", module["core_approach"])

print("\n[12 Resistance Scores]")

display(
    diagnosis["all_scores"][
        [
            "gap",
            "factor",
            "pulse_score",
            "behavior_score",
            "resistance_score"
        ]
    ]
)

INDIVIDUAL AI RESISTANCE DIAGNOSIS
Employee: AP0002
Primary Gap: Habit Gap
Primary Resistance: Switching Cost
Resistance Score: 62.5
Assigned Module: Low-Friction Adoption Module
Core Approach: 마찰 최소화

[12 Resistance Scores]


,gap,factor,pulse_score,behavior_score,resistance_score
0,Habit Gap,Switching Cost,62.5,NaN,62.5000
1,Skill Gap,Task Application Gap,62.5,0.000,43.7500
2,Need Gap,Need Relevance Gap,37.5,NaN,37.5000
3,Need Gap,Value Awareness Gap,37.5,NaN,37.5000
4,Need Gap,Value Experience Gap,37.5,NaN,37.5000
5,Trust Gap,AI Output Trust Gap,37.5,NaN,37.5000
6,Trust Gap,Security Concern,37.5,NaN,37.5000
7,Trust Gap,Responsibility Concern,37.5,NaN,37.5000
8,Skill Gap,Feature Knowledge Gap,37.5,10.000,29.2500
9,Habit Gap,Routine Inertia,25.0,28.125,25.9375


In [14]:
TASK_FILE = "AURA_Task_Context_Synthetic.xlsx"

task_df = pd.read_excel(
    TASK_FILE,
    sheet_name="01_Task_Context"
)

print("Task Context:", task_df.shape)

Task Context: (11626, 18)


In [15]:
def select_current_task(employee_id, task_df):
    """
    직원의 현재 업무 선택.

    우선순위:
    1. In Progress
    2. Planned
    3. 해당 업무 없음 → None
    """

    employee_tasks = task_df[
        task_df["Employee_ID"] == employee_id
    ].copy()

    if employee_tasks.empty:
        return None

    # 1순위: In Progress
    in_progress = employee_tasks[
        employee_tasks["Task_Status"] == "In Progress"
    ]

    if not in_progress.empty:

        # 여러 개면 우선순위가 높은 업무 선택
        priority_order = {
            "High": 1,
            "Medium": 2,
            "Low": 3
        }

        in_progress = in_progress.copy()

        in_progress["priority_rank"] = (
            in_progress["Task_Priority"]
            .map(priority_order)
            .fillna(99)
        )

        return (
            in_progress
            .sort_values("priority_rank")
            .iloc[0]
        )

    # 2순위: Planned
    planned = employee_tasks[
        employee_tasks["Task_Status"] == "Planned"
    ]

    if not planned.empty:
        return planned.iloc[0]

    return None

In [16]:
selected_task = select_current_task(
    employee_id,
    task_df
)

if selected_task is None:
    print("현재 적합한 Task 없음")

else:
    print("선택 Task:", selected_task["Task_Name"])
    print("목적:", selected_task["Task_Purpose"])
    print("Tool:", selected_task["Main_Tool"])
    print("AI Tool:", selected_task["Available_AI_Tool"])
    print("Status:", selected_task["Task_Status"])

선택 Task: 6월 3주차 주간 마케팅 회의 준비
목적: 팀 주간 회의 운영
Tool: Teams
AI Tool: M365 Copilot (Teams 내)
Status: In Progress


In [17]:
MODULE_FILE = "AI_Agent_Module_Library_완성본_1.xlsx"

MODULE_SHEET_MAP = {
    "Need Relevance Gap": "M01 Task Relevance",
    "Value Awareness Gap": "M02 Value Awareness",
    "Value Experience Gap": "M03 Value Experience",
    "Comparative Value Gap": "M04 Comparative Value",

    "Feature Knowledge Gap": "M05 Feature Knowledge",
    "Prompting Gap": "M06 Prompt Coaching",
    "Task Application Gap": "M07 Task Application",

    "AI Output Trust Gap": "M08 Output Trust",
    "Security Concern": "M09 Security Confidence",
    "Responsibility Concern": "M10 Responsibility Clarity",

    "Routine Inertia": "M11 Routine Change",
    "Switching Cost": "M12 Low-Friction Adoption",
}

In [18]:
def load_module_detail(
    primary_resistance,
    module_file=MODULE_FILE
):
    """
    Primary Resistance에 대응하는
    Module 상세설정 로드.
    """

    if primary_resistance not in MODULE_SHEET_MAP:
        raise ValueError(
            f"Module Sheet Mapping 없음: "
            f"{primary_resistance}"
        )

    sheet_name = MODULE_SHEET_MAP[
        primary_resistance
    ]

    module_df = pd.read_excel(
        module_file,
        sheet_name=sheet_name,
        header=1
    )

    module_settings = dict(
        zip(
            module_df["Attribute"],
            module_df["설정값"]
        )
    )

    return {
        "sheet_name": sheet_name,
        "module_id":
            module_settings.get("Module ID"),

        "module_name":
            module_settings.get("Module Name"),

        "module_definition":
            module_settings.get("Module 정의"),

        "objective":
            module_settings.get("Module 목적"),

        "target_behavior":
            module_settings.get("목표 행동"),

        "allowed_intervention":
            module_settings.get("허용 Intervention"),

        "forbidden_intervention":
            module_settings.get("금지 Intervention"),

        "mission_boundary":
            module_settings.get("Mission 범위"),

        "success_criteria":
            module_settings.get("성공 기준")
    }

In [20]:
module_detail = load_module_detail(
    diagnosis["primary_resistance"]
)

for key, value in module_detail.items():
    print(key, ":", value)

sheet_name : M12 Low-Friction Adoption
module_id : M12
module_name : Low-Friction Adoption Module
module_definition : 전환 비용을 최소화한 작은 시도로 도입 장벽을 낮추는 모듈
objective : '바꾸기 번거로움' → 저마찰 작은 성공 경험
target_behavior : 준비 없이 바로 되는 초저비용 AI 활용 1건 수행
allowed_intervention : 원클릭 수준 저마찰 미션
forbidden_intervention : ① 사용자 대신 최종 산출물 전체 대필  ② 업무 외 별도 학습·과제·발표 부여  ③ 허위·과장된 효과 미션  ④ 민감·기밀 정보 입력 유도  ⑤ Target Resistance·Module 임의 변경
mission_boundary : 셋업 불필요한 즉시 활용 1건
success_criteria : 1차: 위 '목표 행동'의 실제 발생·개선  /  2차: Switching Cost를 구성하는 세부 Indicator 개선  /  3차: Re-measurement에서 Switching Cost Score 감소.  ※ 정량 목표치는 PoC 가정값이며 실제 데이터 기반 Calibration 필요.


In [ ]:
import json

def build_agent_input(
    employee_id,
    diagnosis,
    module_detail,
    selected_task,
    data
):
    """
    Diagnosis + Module + Task Context를
    LLM 입력용 JSON 구조로 결합.
    """

    employee_df = data["employees"]

    employee_row = employee_df[
        employee_df["직원ID"] == employee_id
    ]

    if employee_row.empty:
        raise ValueError(
            f"{employee_id}: 직원마스터에 존재하지 않습니다."
        )

    employee = employee_row.iloc[0]

    # Task 없음 처리
    if selected_task is None:
        task_payload = None

    else:
        task_payload = {
            "task_id": selected_task["Task_ID"],
            "task_name": selected_task["Task_Name"],
            "task_purpose": selected_task["Task_Purpose"],
            "task_description": selected_task["Task_Description"],
            "expected_output": selected_task["Expected_Output"],
            "main_tool": selected_task["Main_Tool"],
            "available_ai_tool": selected_task["Available_AI_Tool"],
            "task_status": selected_task["Task_Status"],
            "task_priority": selected_task["Task_Priority"],
            "due_time": str(selected_task["Due_Time"]),
            "collaboration_type": selected_task["Collaboration_Type"],
            "sensitivity_level": selected_task["Sensitivity_Level"]
        }

    agent_input = {

        "employee_context": {
            "employee_id": employee_id,
            "division": employee.get("사업부문"),
            "headquarters": employee.get("본부"),
            "department": employee.get("부서"),
            "team_name": employee.get("팀명"),
            "role_band": employee.get("역할밴드"),
            "power_user": employee.get("Power User")
        },

        "diagnosis": {
            "primary_gap":
                diagnosis["primary_gap"],

            "primary_resistance":
                diagnosis["primary_resistance"],

            "resistance_score":
                diagnosis["resistance_score"]
        },

        "module": {
            "module_id":
                module_detail["module_id"],

            "module_name":
                module_detail["module_name"],

            "module_definition":
                module_detail["module_definition"],

            "objective":
                module_detail["objective"],

            "target_behavior":
                module_detail["target_behavior"],

            "allowed_intervention":
                module_detail["allowed_intervention"],

            "forbidden_intervention":
                module_detail["forbidden_intervention"],

            "mission_boundary":
                module_detail["mission_boundary"],

            "success_criteria":
                module_detail["success_criteria"]
        },

        "task": task_payload,

        "previous_intervention": None
    }

    return agent_input

In [23]:
import json

def build_agent_input(
    employee_id,
    diagnosis,
    module_detail,
    selected_task,
    data
):
    """
    Diagnosis + Module + Task Context를
    LLM 입력용 JSON 구조로 결합.
    """

    employee_df = data["employees"]

    employee_row = employee_df[
        employee_df["직원ID"] == employee_id
    ]

    if employee_row.empty:
        raise ValueError(
            f"{employee_id}: 직원마스터에 존재하지 않습니다."
        )

    employee = employee_row.iloc[0]

    if selected_task is None:
        task_payload = None
    else:
        task_payload = {
            "task_id": selected_task["Task_ID"],
            "task_name": selected_task["Task_Name"],
            "task_purpose": selected_task["Task_Purpose"],
            "task_description": selected_task["Task_Description"],
            "expected_output": selected_task["Expected_Output"],
            "main_tool": selected_task["Main_Tool"],
            "available_ai_tool": selected_task["Available_AI_Tool"],
            "task_status": selected_task["Task_Status"],
            "task_priority": selected_task["Task_Priority"],
            "due_time": str(selected_task["Due_Time"]),
            "collaboration_type": selected_task["Collaboration_Type"],
            "sensitivity_level": selected_task["Sensitivity_Level"]
        }

    agent_input = {
        "employee_context": {
            "employee_id": employee_id,
            "division": employee.get("사업부문"),
            "headquarters": employee.get("본부"),
            "department": employee.get("부서"),
            "team_name": employee.get("팀명"),
            "role_band": employee.get("역할밴드"),
            "power_user": employee.get("Power User")
        },

        "diagnosis": {
            "primary_gap": diagnosis["primary_gap"],
            "primary_resistance": diagnosis["primary_resistance"],
            "resistance_score": diagnosis["resistance_score"]
        },

        "module": {
            "module_id": module_detail["module_id"],
            "module_name": module_detail["module_name"],
            "module_definition": module_detail["module_definition"],
            "objective": module_detail["objective"],
            "target_behavior": module_detail["target_behavior"],
            "allowed_intervention": module_detail["allowed_intervention"],
            "forbidden_intervention": module_detail["forbidden_intervention"],
            "mission_boundary": module_detail["mission_boundary"],
            "success_criteria": module_detail["success_criteria"]
        },

        "task": task_payload,
        "previous_intervention": None
    }

    return agent_input

In [24]:
agent_input = build_agent_input(
    employee_id,
    diagnosis,
    module_detail,
    selected_task,
    data
)

print(
    json.dumps(
        agent_input,
        ensure_ascii=False,
        indent=2
    )
)

{
  "employee_context": {
    "employee_id": "AP0002",
    "division": "화장품/생활용품",
    "headquarters": "럭셔리브랜드",
    "department": "설화수브랜드",
    "team_name": "설화수브랜드 1팀",
    "role_band": "Expert",
    "power_user": "N"
  },
  "diagnosis": {
    "primary_gap": "Habit Gap",
    "primary_resistance": "Switching Cost",
    "resistance_score": 62.5
  },
  "module": {
    "module_id": "M12",
    "module_name": "Low-Friction Adoption Module",
    "module_definition": "전환 비용을 최소화한 작은 시도로 도입 장벽을 낮추는 모듈",
    "objective": "'바꾸기 번거로움' → 저마찰 작은 성공 경험",
    "target_behavior": "준비 없이 바로 되는 초저비용 AI 활용 1건 수행",
    "allowed_intervention": "원클릭 수준 저마찰 미션",
    "forbidden_intervention": "① 사용자 대신 최종 산출물 전체 대필  ② 업무 외 별도 학습·과제·발표 부여  ③ 허위·과장된 효과 미션  ④ 민감·기밀 정보 입력 유도  ⑤ Target Resistance·Module 임의 변경",
    "mission_boundary": "셋업 불필요한 즉시 활용 1건",
    "success_criteria": "1차: 위 '목표 행동'의 실제 발생·개선  /  2차: Switching Cost를 구성하는 세부 Indicator 개선  /  3차: Re-measurement에서 Switching Cost Score 감소.  ※ 정량 목표치는 PoC 가정값

In [50]:
from openai import OpenAI
from pydantic import BaseModel
from typing import Optional
import json

client = OpenAI()


class MissionOutput(BaseModel):
    response_status: str
    target_type: str
    module_id: str
    target_resistance: str

    selected_task_id: Optional[str] = None
    mission_title: Optional[str] = None

    # 직원 UI용
    popup_message: Optional[str] = None
    popup_reason: Optional[str] = None

    # 시스템/로그용 상세 내용
    target_work: Optional[str] = None
    mission_action: Optional[str] = None
    execution_timing: Optional[str] = None
    execution_method: Optional[str] = None
    ai_tool: Optional[str] = None
    completion_criteria: Optional[str] = None
    estimated_burden: Optional[str] = None
    mission_rationale: Optional[str] = None
    verification_method: Optional[str] = None
    feedback_rule: Optional[str] = None
    next_action: Optional[str] = None

    no_mission_reason: Optional[str] = None

In [59]:
SYSTEM_PROMPT = """
너는 기업 구성원의 AI 활용 행동을 변화시키는
AI Intervention Agent다.

반드시 다음 규칙을 지킨다.

1. Input의 Primary Resistance를 절대 변경하지 않는다.
2. Input의 Module을 절대 변경하지 않는다.
3. Module의 목적, 목표 행동, 허용 Intervention,
   금지 Intervention, Mission Boundary를 준수한다.
4. 사용자가 현재 실제로 수행 중인 Task 안에서만 개입한다.
5. 업무 외 별도 교육, 과제, 발표를 만들지 않는다.
6. 한 번에 하나의 작은 행동 변화를 우선한다.
7. Input에 명시된 Available AI Tool만 사용한다.
8. Mission은 무엇을, 언제, 어떻게 수행할지 구체적으로 작성한다.
9. 완료 여부를 관찰할 수 있는 Completion Criteria를 제시한다.
10. 적절한 Mission이 없으면 억지로 만들지 말고 NO_MISSION을 반환한다.
11. 민감·기밀 정보 입력을 유도하지 않는다.
12. 최종 업무 산출물을 사용자 대신 전부 작성하지 않는다.

[직원용 Popup 규칙]

13. popup_message는 일반 직원이 읽는 짧은 행동 제안이다.
14. popup_message는 최대 2문장으로 작성한다.
15. popup_message에는 Resistance, Module, Score 등
    진단 시스템 용어를 절대 노출하지 않는다.
16. popup_message에는 기술적인 설명보다
    "지금 무엇을 하면 되는지"만 쉽게 표현한다.
17. popup_reason은 최대 1문장으로 작성한다.
18. popup_reason은 직원에게 부담을 주지 않는 표현을 사용한다.
19. WHEN, AI TOOL, DONE WHEN 같은 필드명을 popup_message에 쓰지 않는다.
20. popup_message와 popup_reason은
    친근하지만 업무용으로 자연스러운 문체를 사용한다.

response_status는 반드시 아래 둘 중 하나다.
- MISSION_GENERATED
- NO_MISSION

target_type은 INDIVIDUAL로 반환한다.

estimated_burden은 Mission 생성 시
LOW 또는 MEDIUM 중 하나로 한다.
"""

In [60]:
response = client.responses.parse(
    model="gpt-5.6-terra",
    input=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": json.dumps(
                agent_input,
                ensure_ascii=False,
                default=str
            )
        }
    ],
    text_format=MissionOutput
)

mission = response.output_parsed

print(
    mission.model_dump_json(
        indent=2
    )
)

{
  "response_status": "MISSION_GENERATED",
  "target_type": "INDIVIDUAL",
  "module_id": "M12",
  "target_resistance": "Switching Cost",
  "selected_task_id": "TSK000004",
  "mission_title": "이전 회의록에서 미결 액션만 빠르게 추리기",
  "popup_message": "회의 준비를 시작할 때 Teams의 이전 회의록을 열고, Copilot에게 이번 주 확인할 미결 액션만 항목으로 정리해 달라고 요청해 보세요. 나온 내용 중 필요한 항목만 회의록 초안에 옮기면 됩니다.",
  "popup_reason": "기존 회의록을 그대로 활용하면 새로 정리하는 수고를 줄일 수 있습니다.",
  "target_work": "6월 3주차 주간 마케팅 회의 준비를 위한 회의록 초안 정리",
  "mission_action": "Teams의 직전 주간 회의록에서 Copilot으로 미결 액션 항목만 추려, 이번 회의록 초안의 확인 안건으로 반영한다.",
  "execution_timing": "회의록 초안 작성을 시작하는 즉시, 직전 회의록을 열었을 때 1회 수행",
  "execution_method": "Teams 내 직전 회의록 또는 회의 대화의 공개 범위 내 내용만 대상으로 Copilot에 “이 회의록에서 아직 완료 여부를 확인해야 할 액션 항목만 담당자·할 일·기한이 있으면 함께 글머리표로 정리해줘”라고 요청한다. 결과를 확인한 뒤 필요한 항목만 직접 회의록 초안에 추가한다.",
  "ai_tool": "M365 Copilot (Teams 내)",
  "completion_criteria": "Copilot으로 직전 회의록의 미결 액션 항목 목록을 1회 생성하고, 검토 후 그중 필요한 항목 1개 이상을 이번 주 회의록 초안의 확인 안건으로 반영했다.",
  "estimated_burden": "LOW",
  "miss

In [61]:
def validate_mission_output(
    mission,
    agent_input
):
    """
    LLM Mission이 Input의 진단/Module/Task 범위를
    벗어나지 않았는지 검증.
    """

    errors = []

    # 1. Module ID 일치
    if mission.module_id != agent_input["module"]["module_id"]:
        errors.append(
            f"Module ID mismatch: "
            f"{mission.module_id} != "
            f"{agent_input['module']['module_id']}"
        )

    # 2. Resistance 일치
    if (
        mission.target_resistance
        != agent_input["diagnosis"]["primary_resistance"]
    ):
        errors.append(
            f"Resistance mismatch: "
            f"{mission.target_resistance} != "
            f"{agent_input['diagnosis']['primary_resistance']}"
        )

    # 3. target_type
    if mission.target_type != "INDIVIDUAL":
        errors.append(
            f"Invalid target_type: {mission.target_type}"
        )

    # 4. response_status
    allowed_status = [
        "MISSION_GENERATED",
        "NO_MISSION"
    ]

    if mission.response_status not in allowed_status:
        errors.append(
            f"Invalid response_status: "
            f"{mission.response_status}"
        )

    # -------------------------------------------------
    # NO_MISSION일 때
    # -------------------------------------------------

    if mission.response_status == "NO_MISSION":

        if not mission.no_mission_reason:
            errors.append(
                "NO_MISSION인데 no_mission_reason이 없습니다."
            )

        return errors

    # -------------------------------------------------
    # MISSION_GENERATED일 때
    # -------------------------------------------------

    task = agent_input["task"]

    if task is None:
        errors.append(
            "Task가 없는데 Mission이 생성되었습니다."
        )

        return errors

    # 5. Task ID 검증
    if mission.selected_task_id != task["task_id"]:
        errors.append(
            f"Task ID mismatch: "
            f"{mission.selected_task_id} != "
            f"{task['task_id']}"
        )

    # 6. AI Tool 검증
    if mission.ai_tool != task["available_ai_tool"]:
        errors.append(
            f"AI Tool mismatch: "
            f"{mission.ai_tool} != "
            f"{task['available_ai_tool']}"
        )

    # 7. 완료 기준
    if not mission.completion_criteria:
        errors.append(
            "completion_criteria가 없습니다."
        )

    # 8. 검증 방법
    if not mission.verification_method:
        errors.append(
            "verification_method가 없습니다."
        )

    # 9. 부담 수준
    if mission.estimated_burden not in [
        "LOW",
        "MEDIUM"
    ]:
        errors.append(
            f"Invalid estimated_burden: "
            f"{mission.estimated_burden}"
        )

    return errors

In [62]:
validation_errors = validate_mission_output(
    mission,
    agent_input
)

if len(validation_errors) == 0:
    print("✅ Mission Validation PASS")

else:
    print("❌ Mission Validation FAIL")

    for error in validation_errors:
        print("-", error)

✅ Mission Validation PASS


In [63]:
print("=== API CALL CHECK ===")
print("Response ID :", response.id)
print("Model       :", response.model)
print("Input tokens :", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)
print("Total tokens :", response.usage.total_tokens)

=== API CALL CHECK ===
Response ID : resp_00eef8e871ec59b4006a98dad4d29887d0af036a603df52389
Model       : gpt-5.6-terra
Input tokens : 1455
Output tokens: 764
Total tokens : 2219


In [64]:
import tkinter as tk
from datetime import datetime
import csv
import os


def save_intervention_action(
    employee_id,
    mission,
    action,
    file_name="intervention_action_log.csv"
):
    """
    Mission에 대한 사용자 행동 로그 저장
    """

    file_exists = os.path.exists(file_name)

    with open(
        file_name,
        "a",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        writer = csv.writer(f)

        if not file_exists:
            writer.writerow([
                "employee_id",
                "module_id",
                "target_resistance",
                "selected_task_id",
                "mission_title",
                "action",
                "timestamp"
            ])

        writer.writerow([
            employee_id,
            mission.module_id,
            mission.target_resistance,
            mission.selected_task_id,
            mission.mission_title,
            action,
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        ])

In [67]:
response = client.responses.parse(
    model="gpt-5.6-terra",
    input=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": json.dumps(
                agent_input,
                ensure_ascii=False,
                default=str
            )
        }
    ],
    text_format=MissionOutput
)

mission = response.output_parsed

print(
    mission.model_dump_json(
        indent=2
    )
)

{
  "response_status": "MISSION_GENERATED",
  "target_type": "INDIVIDUAL",
  "module_id": "M12",
  "target_resistance": "Switching Cost",
  "selected_task_id": "TSK000004",
  "mission_title": "회의 전 확인 질문 3개 받기",
  "popup_message": "회의 준비 중 Teams의 Copilot에 이번 회의에서 확인할 결정사항·담당자 질문 3개를 요청해 보세요. 나온 질문 중 1개만 회의록 준비 메모에 반영하면 됩니다.",
  "popup_reason": "회의 준비 흐름을 바꾸지 않고 바로 한 번 활용해 볼 수 있습니다.",
  "target_work": "6월 3주차 주간 마케팅 회의의 회의록 준비",
  "mission_action": "Teams 내 M365 Copilot으로 회의에서 확인할 질문 3개를 생성하고, 그중 1개를 회의록 준비 메모에 반영한다.",
  "execution_timing": "지금 회의록을 준비하는 중, 회의 시작 전",
  "execution_method": "Teams에서 M365 Copilot을 열고 공개 가능한 회의 목적 범위에서 “이번 주간 마케팅 회의에서 확인할 결정사항과 담당자 확인 질문을 3개로 정리해줘”라고 요청한다. 응답 중 가장 적합한 질문 1개를 회의록 준비 메모에 추가한다. 민감하거나 기밀인 세부 내용은 입력하지 않는다.",
  "ai_tool": "M365 Copilot (Teams 내)",
  "completion_criteria": "Copilot에 질문 1회를 실행했고, 생성된 질문 3개 중 1개가 해당 회의의 회의록 준비 메모에 반영되어 있다.",
  "estimated_burden": "LOW",
  "mission_rationale": "기존 Teams 작업 화면에서 짧은 요청 한 번만 수행해 AI 활용의 전환 부담을 낮춘다.",
  "

In [68]:
import tkinter as tk
from datetime import datetime
import csv
import os


# ============================================================
# 1. ACTION LOG 저장
# ============================================================

def save_intervention_action(
    employee_id,
    mission,
    action,
    file_name="intervention_action_log.csv"
):
    """
    직원이 팝업에서 누른 행동을 CSV로 저장.
    """

    file_exists = os.path.exists(file_name)

    with open(
        file_name,
        "a",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        writer = csv.writer(f)

        if not file_exists:
            writer.writerow([
                "employee_id",
                "module_id",
                "target_resistance",
                "selected_task_id",
                "mission_title",
                "action",
                "timestamp"
            ])

        writer.writerow([
            employee_id,
            mission.module_id,
            mission.target_resistance,
            mission.selected_task_id,
            mission.mission_title,
            action,
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        ])


# ============================================================
# 2. 직원용 Mission Popup
# ============================================================

def show_individual_mission_popup(
    employee_id,
    mission
):
    """
    일반 직원에게 보여주는 Daily Mission 팝업.
    """

    # --------------------------------------------------------
    # Mission 미생성 처리
    # --------------------------------------------------------

    if mission.response_status != "MISSION_GENERATED":

        print(
            "Mission 미생성:",
            mission.no_mission_reason
        )

        return


    # ========================================================
    # 직원 화면용 텍스트
    # ========================================================

    popup_title = "오늘 업무, Copilot과 함께 해볼까요?"

    mission_title = (
        mission.mission_title
        or "오늘의 AI Mission"
    )

    popup_message = (
        mission.popup_message
        or mission.mission_action
        or "현재 업무에서 Copilot을 한 번 활용해보세요."
    )

    popup_reason = (
        mission.popup_reason
        or "지금 하고 있는 업무 안에서 바로 적용할 수 있어요."
    )


    # ========================================================
    # 버튼 이벤트
    # ========================================================

    def start_mission():

        save_intervention_action(
            employee_id,
            mission,
            "STARTED"
        )

        start_button.config(
            text="✓ 시작했어요",
            bg="#187E72",
            state="disabled"
        )

        later_button.config(
            state="disabled"
        )

        root.after(
            900,
            root.destroy
        )


    def postpone_mission():

        save_intervention_action(
            employee_id,
            mission,
            "POSTPONED"
        )

        later_button.config(
            text="나중에 확인할게요",
            state="disabled"
        )

        start_button.config(
            state="disabled"
        )

        root.after(
            900,
            root.destroy
        )


    # ========================================================
    # Window
    # ========================================================

    root = tk.Tk()

    root.title("AI Work Assistant")

    WINDOW_W = 430
    WINDOW_H = 300

    root.geometry(
        f"{WINDOW_W}x{WINDOW_H}"
    )

    root.configure(
        bg="#F4F6F8"
    )

    root.resizable(
        False,
        False
    )

    root.attributes(
        "-topmost",
        True
    )


    # ========================================================
    # 우측 하단 배치
    # ========================================================

    root.update_idletasks()

    screen_w = root.winfo_screenwidth()
    screen_h = root.winfo_screenheight()

    x = screen_w - WINDOW_W - 30
    y = screen_h - WINDOW_H - 70

    root.geometry(
        f"{WINDOW_W}x{WINDOW_H}+{x}+{y}"
    )


    # ========================================================
    # Main Card
    # ========================================================

    card = tk.Frame(
        root,
        bg="white",
        highlightthickness=1,
        highlightbackground="#E3E8EC"
    )

    card.pack(
        fill="both",
        expand=True,
        padx=10,
        pady=10
    )


    # ========================================================
    # Header
    # ========================================================

    header = tk.Frame(
        card,
        bg="white"
    )

    header.pack(
        fill="x",
        padx=22,
        pady=(18, 0)
    )


    icon = tk.Label(
        header,
        text="✦",
        bg="#E9F6F5",
        fg="#15949C",
        font=("맑은 고딕", 10, "bold"),
        width=2
    )

    icon.pack(
        side="left"
    )


    brand = tk.Label(
        header,
        text="  AI Work Assistant",
        bg="white",
        fg="#17344B",
        font=("맑은 고딕", 9, "bold")
    )

    brand.pack(
        side="left"
    )


    time_label = tk.Label(
        header,
        text="지금",
        bg="white",
        fg="#A0AAB2",
        font=("맑은 고딕", 8)
    )

    time_label.pack(
        side="right"
    )


    # ========================================================
    # Main Title
    # ========================================================

    title_label = tk.Label(
        card,
        text=popup_title,
        bg="white",
        fg="#17324A",
        font=("맑은 고딕", 13, "bold"),
        justify="left",
        anchor="w"
    )

    title_label.pack(
        anchor="w",
        padx=22,
        pady=(15, 3)
    )


    # ========================================================
    # Mission Title
    # ========================================================

    mission_title_label = tk.Label(
        card,
        text=mission_title,
        bg="white",
        fg="#81909B",
        font=("맑은 고딕", 8),
        justify="left",
        anchor="w",
        wraplength=370
    )

    mission_title_label.pack(
        anchor="w",
        padx=22,
        pady=(0, 10)
    )


    # ========================================================
    # Main Mission Message
    # ========================================================

    mission_box = tk.Frame(
        card,
        bg="#F6F9FA"
    )

    mission_box.pack(
        fill="x",
        padx=22
    )


    mission_label = tk.Label(
        mission_box,
        text=popup_message,
        bg="#F6F9FA",
        fg="#1C3548",
        font=("맑은 고딕", 10, "bold"),
        justify="left",
        anchor="w",
        wraplength=340,
        padx=14,
        pady=12
    )

    mission_label.pack(
        fill="x"
    )


    # ========================================================
    # Reason / Tip
    # ========================================================

    reason_label = tk.Label(
        card,
        text="TIP  " + popup_reason,
        bg="white",
        fg="#75848F",
        font=("맑은 고딕", 8),
        justify="left",
        anchor="w",
        wraplength=365
    )

    reason_label.pack(
        anchor="w",
        padx=22,
        pady=(10, 5)
    )


    # ========================================================
    # Buttons
    # ========================================================

    button_frame = tk.Frame(
        card,
        bg="white"
    )

    button_frame.pack(
        side="bottom",
        fill="x",
        padx=22,
        pady=(8, 18)
    )


    later_button = tk.Button(
        button_frame,

        text="나중에",

        command=postpone_mission,

        bg="#EEF1F3",
        fg="#536471",

        activebackground="#E6EAED",

        relief="flat",
        bd=0,

        font=("맑은 고딕", 9),

        height=2,

        cursor="hand2"
    )

    later_button.pack(
        side="left",
        fill="x",
        expand=True,
        padx=(0, 6)
    )


    start_button = tk.Button(
        button_frame,

        text="지금 해보기",

        command=start_mission,

        bg="#173A56",
        fg="white",

        activebackground="#224B69",
        activeforeground="white",

        relief="flat",
        bd=0,

        font=("맑은 고딕", 9, "bold"),

        height=2,

        cursor="hand2"
    )

    start_button.pack(
        side="left",
        fill="x",
        expand=True,
        padx=(6, 0)
    )


    # ========================================================
    # Popup 실행
    # ========================================================

    root.mainloop()

In [69]:
show_individual_mission_popup(
    employee_id,
    mission
)

In [71]:
import os

log_file = "intervention_action_log.csv"

if os.path.exists(log_file):
    os.remove(log_file)
    print("기존 로그 파일 삭제 완료")
else:
    print("기존 로그 파일 없음")

기존 로그 파일 삭제 완료


In [72]:
show_individual_mission_popup(
    employee_id,
    mission
)

In [73]:
import pandas as pd

log_df = pd.read_csv(
    "intervention_action_log.csv",
    encoding="utf-8-sig"
)

display(log_df)

,employee_id,module_id,target_resistance,selected_task_id,mission_title,action,timestamp
0,AP0002,M12,Switching Cost,TSK000004,회의 전 확인 질문 3개 받기,STARTED,2026-09-03 11:33:20


In [74]:
def run_individual_pipeline(
    employee_id,
    data,
    task_df,
    show_popup=True
):
    """
    Individual AI Intervention End-to-End Pipeline

    1. Individual Resistance 진단
    2. Primary Resistance 자동 선정
    3. Module 자동 매칭
    4. Module 상세정보 로드
    5. 현재 Task 선택
    6. Agent Input 생성
    7. OpenAI API Mission 생성
    8. Mission Validation
    9. 직원용 Popup 표시
    """

    print("=" * 55)
    print("INDIVIDUAL AI INTERVENTION PIPELINE")
    print("=" * 55)

    # --------------------------------------------------
    # 1. Diagnosis
    # --------------------------------------------------

    diagnosis = diagnose_individual(
        employee_id,
        data
    )

    print(
        f"[1] Diagnosis        : "
        f"{diagnosis['primary_resistance']} "
        f"({diagnosis['resistance_score']})"
    )


    # --------------------------------------------------
    # 2. Module Matching
    # --------------------------------------------------

    module = match_module(
        diagnosis["primary_resistance"],
        data
    )

    print(
        f"[2] Module           : "
        f"{module['module_name']}"
    )


    # --------------------------------------------------
    # 3. Module Detail
    # --------------------------------------------------

    module_detail = load_module_detail(
        diagnosis["primary_resistance"]
    )

    print(
        f"[3] Module Detail    : "
        f"{module_detail['module_id']}"
    )


    # --------------------------------------------------
    # 4. Current Task
    # --------------------------------------------------

    selected_task = select_current_task(
        employee_id,
        task_df
    )

    if selected_task is None:

        print(
            "[4] Current Task     : 없음"
        )

    else:

        print(
            f"[4] Current Task     : "
            f"{selected_task['Task_Name']}"
        )


    # --------------------------------------------------
    # 5. Agent Input
    # --------------------------------------------------

    agent_input = build_agent_input(
        employee_id,
        diagnosis,
        module_detail,
        selected_task,
        data
    )

    print(
        "[5] Agent Input      : Generated"
    )


    # --------------------------------------------------
    # 6. OpenAI API
    # --------------------------------------------------

    response = client.responses.parse(
        model="gpt-5.6-terra",
        input=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": json.dumps(
                    agent_input,
                    ensure_ascii=False,
                    default=str
                )
            }
        ],
        text_format=MissionOutput
    )

    mission = response.output_parsed

    print(
        f"[6] Mission          : "
        f"{mission.response_status}"
    )


    # --------------------------------------------------
    # 7. Validation
    # --------------------------------------------------

    validation_errors = validate_mission_output(
        mission,
        agent_input
    )

    if validation_errors:

        print(
            "[7] Validation       : FAIL"
        )

        for error in validation_errors:
            print(
                "    -",
                error
            )

        return {
            "success": False,
            "employee_id": employee_id,
            "diagnosis": diagnosis,
            "module": module,
            "module_detail": module_detail,
            "selected_task": selected_task,
            "agent_input": agent_input,
            "mission": mission,
            "validation_errors": validation_errors
        }


    print(
        "[7] Validation       : PASS"
    )


    # --------------------------------------------------
    # 8. API Usage
    # --------------------------------------------------

    print(
        f"[8] API Usage        : "
        f"{response.usage.total_tokens} tokens"
    )


    # --------------------------------------------------
    # 9. Mission Summary
    # --------------------------------------------------

    if mission.response_status == "MISSION_GENERATED":

        print(
            f"[9] Popup Message    : "
            f"{mission.popup_message}"
        )

    else:

        print(
            f"[9] No Mission       : "
            f"{mission.no_mission_reason}"
        )


    # --------------------------------------------------
    # 10. Popup
    # --------------------------------------------------

    if (
        show_popup
        and
        mission.response_status == "MISSION_GENERATED"
    ):

        show_individual_mission_popup(
            employee_id,
            mission
        )


    # --------------------------------------------------
    # Result
    # --------------------------------------------------

    return {
        "success": True,
        "employee_id": employee_id,
        "diagnosis": diagnosis,
        "module": module,
        "module_detail": module_detail,
        "selected_task": selected_task,
        "agent_input": agent_input,
        "mission": mission,
        "api_response": response,
        "validation_errors": []
    }

In [75]:
result = run_individual_pipeline(
    "AP0002",
    data,
    task_df
)

INDIVIDUAL AI INTERVENTION PIPELINE
[1] Diagnosis        : Switching Cost (62.5)
[2] Module           : Low-Friction Adoption Module
[3] Module Detail    : M12
[4] Current Task     : 6월 3주차 주간 마케팅 회의 준비
[5] Agent Input      : Generated
[6] Mission          : MISSION_GENERATED
[7] Validation       : PASS
[8] API Usage        : 1982 tokens
[9] Popup Message    : 회의 준비 중 Teams에서 Copilot에게 안건을 3개로 정리해 달라고 요청해 보세요. 나온 초안에서 필요한 표현만 직접 다듬어 회의록에 넣으면 됩니다.


In [77]:
result = run_individual_pipeline(
    "AP0003",
    data,
    task_df
)

INDIVIDUAL AI INTERVENTION PIPELINE
[1] Diagnosis        : Comparative Value Gap (50.0)
[2] Module           : Comparative Value Module
[3] Module Detail    : M04
[4] Current Task     : 소비자 반응·리뷰 분석
[5] Agent Input      : Generated
[6] Mission          : MISSION_GENERATED
[7] Validation       : PASS
[8] API Usage        : 2151 tokens
[9] Popup Message    : 지금 분석 중인 리뷰 10건을 골라, 먼저 평소 방식으로 핵심 불만 유형을 분류해 보세요. 같은 10건을 Excel 내 Copilot으로도 분류한 뒤 시간과 수정 필요 여부만 비교해 보세요.
